[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C08_Training_Systems_Course/03_parallelism_strategies/03_parallelism_strategies.ipynb)

# 03 · 并行策略：DP/TP/PP/ZeRO/FSDP

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
在真实模型规模上算 ZeRO 各阶段每卡显存、pipeline 气泡、DP 通信量，并判断"训这个模型至少要几张卡"。

**你将完成：**
1. ZeRO-1/2/3 的每卡显存（分片优化器/梯度/参数）
2. pipeline 气泡占比与 micro-batch 数的关系
3. DP all-reduce 通信量
4. 训 pythia-12b 至少需要几张 80GB A100（ZeRO-3）

> 数据：真实 Pythia / GPT-NeoX config + GPU 规格。

## 0 · config 管线

In [ ]:
import os, json, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.training_systems_data"); os.makedirs(CACHE, exist_ok=True)
MODELS={"pythia-1.4b":"https://huggingface.co/EleutherAI/pythia-1.4b/resolve/main/config.json",
        "pythia-6.9b":"https://huggingface.co/EleutherAI/pythia-6.9b/resolve/main/config.json",
        "pythia-12b":"https://huggingface.co/EleutherAI/pythia-12b/resolve/main/config.json",
        "gpt-neox-20b":"https://huggingface.co/EleutherAI/gpt-neox-20b/resolve/main/config.json"}
def load_config(m):
    p=os.path.join(CACHE,f"{m}.json")
    if not os.path.exists(p): urllib.request.urlretrieve(MODELS[m],p)
    c=json.load(open(p)); g=lambda *k: next(c[x] for x in k if x in c); h=g("hidden_size","n_embd")
    return dict(L=g("num_hidden_layers","n_layer"),h=h,heads=g("num_attention_heads","n_head"),
                V=g("vocab_size"),I=c.get("intermediate_size",4*h))
def param_count(c):
    L,h,V,I=c["L"],c["h"],c["V"],c["I"]; return 2*V*h+L*(4*h*h+4*h+2*h*I+(I+h)+4*h)+h
GB=1024**3
print('ok')

## 1 · ZeRO 各阶段每卡显存

DP：每卡都存完整 16P。ZeRO 把优化器(12P)/梯度(2P)/参数(2P)逐步分片到 N 卡。

In [ ]:
def per_gpu_gb(P, N, stage):
    # 16P = 权重2P + 梯度2P + 优化器12P
    if stage==0:   m = 16*P                       # 纯 DP，不分片
    elif stage==1: m = 2*P + 2*P + 12*P/N         # 分优化器
    elif stage==2: m = 2*P + (2*P+12*P)/N         # +分梯度
    elif stage==3: m = 16*P/N                      # +分参数
    return m/GB
P=param_count(load_config("pythia-12b"))
print(f"pythia-12b ({P/1e9:.1f}B) 在 N=8 卡上每卡训练状态显存:")
for s in [0,1,2,3]:
    print(f"  ZeRO-{s}: {per_gpu_gb(P,8,s):6.1f} GB/卡")
print("\n=> ZeRO-3 把每卡显存压到 ~1/N，这是用中等卡训大模型的关键")

## 2 · Pipeline 气泡

气泡占比 = (p-1)/(m+p-1)。增大 micro-batch 数 m 摊薄气泡。

In [ ]:
def bubble_fraction(p, m): return (p-1)/(m+p-1)
p=8
print(f"流水线段数 p={p}:")
for m in [1,4,8,32,128]:
    print(f"  micro-batch m={m:3d}: 气泡占比 {bubble_fraction(p,m):.1%}  (有效利用率 {1-bubble_fraction(p,m):.1%})")
print("\n=> m 太小气泡巨大；要 m >> p 才划算。这就是 PP 总配大 micro-batch 的原因")

## 3 · DP all-reduce 通信量

ring all-reduce 每步通信 ≈ 2P·字节，与卡数无关。

In [ ]:
def allreduce_bytes(P, dtype_bytes=2): return 2*P*dtype_bytes
for m in ["pythia-1.4b","pythia-12b"]:
    P=param_count(load_config(m))
    vol=allreduce_bytes(P)
    # 在 200 GB/s 互联上的通信时间
    t=vol/200e9
    print(f"{m:12s} 每步 all-reduce {vol/GB:.1f} GB  在200GB/s互联上 ≈ {t*1000:.0f} ms")
print("\n=> 通信量与卡数无关（ring 的妙处），但和模型大小成正比")

## 4 · 训 pythia-12b 至少几张 80GB A100

用 ZeRO-3，每卡 16P/N + activation，要 <= 80GB。

In [ ]:
def min_gpus_zero3(P, card_gb, act_gb=10):
    N=1
    while (16*P/N)/GB + act_gb > card_gb: N*=2
    return N
P=param_count(load_config("pythia-12b"))
N=min_gpus_zero3(P, 80)
print(f"pythia-12b ZeRO-3 训练: 至少 {N} 张 80GB A100 (每卡 {16*P/N/GB:.0f}GB状态 + activation)")
P20=param_count(load_config("gpt-neox-20b"))
print(f"gpt-neox-20b ZeRO-3 训练: 至少 {min_gpus_zero3(P20,80)} 张 80GB A100")

## 5 · DP vs ZeRO-3 每卡显存对比

把模块讲解的核心洞见做实：纯数据并行(DP)每张卡都背一份完整的 16P 状态（与卡数无关的重复拷贝），而 ZeRO-3 把它分片成 16P/N。固定模型、变卡数，看分片到底省多少、为什么 ZeRO-3 能训 DP 放不下的模型。

In [ ]:
# 固定卡数，对比 DP(stage0) 与 ZeRO-3 的每卡显存：分片省多少？
P=param_count(load_config("pythia-12b"))
print(f"pythia-12b ({P/1e9:.1f}B) 训练状态(不含 activation)每卡显存:")
print(f"{'卡数 N':>6s} {'DP(每卡16P)':>14s} {'ZeRO-3(16P/N)':>16s} {'省下倍数':>10s}")
for N in [8, 16, 32, 64]:
    dp=per_gpu_gb(P, N, 0); z3=per_gpu_gb(P, N, 3)
    print(f"{N:6d} {dp:13.0f}G {z3:15.1f}G {dp/z3:9.0f}x")
# 自检：DP 与卡数无关(每卡都存完整 16P)；ZeRO-3 随 N 线性下降
assert abs(per_gpu_gb(P,8,0) - per_gpu_gb(P,64,0)) < 1e-6, "DP 每卡显存与卡数无关"
assert abs(per_gpu_gb(P,16,3) - per_gpu_gb(P,8,3)/2) < 1e-3, "ZeRO-3 翻倍卡数则每卡减半"
assert per_gpu_gb(P,8,3) < per_gpu_gb(P,8,0), "ZeRO-3 远比 DP 省"
print("\n=> DP 每卡都背完整 16P(纯浪费的重复拷贝)；ZeRO-3 把它分片成 16P/N，")
print("   这就是为什么同样一堆卡，ZeRO-3 能训 DP 完全放不下的模型")

---
## ✏️ 练习区

### ✏️ 练习 1：ZeRO 每卡显存

实现 `zero_per_gpu_gb(P, N, stage)`（stage∈{1,2,3}），返回每卡训练状态显存(GB)。
公式见正文。

In [ ]:
def zero_per_gpu_gb(P, N, stage):
    # TODO: stage1: 2P+2P+12P/N ; stage2: 2P+(2P+12P)/N ; stage3: 16P/N （再 /GB）
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
P=param_count(load_config("pythia-6.9b"))
g1,g2,g3 = [zero_per_gpu_gb(P,8,s) for s in (1,2,3)]
assert g1 > g2 > g3, "阶段越高分片越多，每卡越省"
assert abs(g3 - 16*P/8/GB) < 1e-6
# N 越大 stage3 越省
assert zero_per_gpu_gb(P,16,3) < zero_per_gpu_gb(P,8,3)
print(f"练习 1 通过 ✓  6.9b@8卡: ZeRO-1={g1:.0f}G ZeRO-2={g2:.0f}G ZeRO-3={g3:.0f}G")


### ✏️ 练习 2：pipeline 气泡

实现 `pipeline_bubble(p, m)` 返回气泡占比。再实现 `microbatch_for_target(p, target)`：
返回让气泡 <= target 所需的最小 m。

In [ ]:
def pipeline_bubble(p, m):
    # TODO: (p-1)/(m+p-1)
    raise NotImplementedError
def microbatch_for_target(p, target):
    # TODO: 最小 m 使 bubble <= target
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
assert abs(pipeline_bubble(4,4) - 3/7) < 1e-9
assert pipeline_bubble(8,1) > pipeline_bubble(8,100), "m 越大气泡越小"
m=microbatch_for_target(8, 0.1)
assert pipeline_bubble(8,m) <= 0.1 and pipeline_bubble(8,m-1) > 0.1
print(f"练习 2 通过 ✓  p=8 要把气泡压到10%需 m>={m}")


### ✏️ 练习 3：DP 通信时间

实现 `allreduce_time_ms(P, bw_gbps, dtype_bytes)`：ring all-reduce 通信 2P·bytes / 带宽，返回毫秒。

In [ ]:
def allreduce_time_ms(P, bw_gbps, dtype_bytes=2):
    # TODO: 2*P*dtype_bytes / (bw_gbps*1e9) * 1000
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
P=param_count(load_config("pythia-1.4b"))
t_fast=allreduce_time_ms(P, 600)   # NVLink
t_slow=allreduce_time_ms(P, 25)    # 慢以太网
assert t_slow > t_fast and abs(t_fast - 2*P*2/600e9*1000) < 1e-6
print(f"练习 3 通过 ✓  1.4b all-reduce: NVLink {t_fast:.1f}ms vs 25Gbps {t_slow:.0f}ms")


### ✏️ 练习 4：训这个模型至少几张卡

实现 `min_gpus(P, card_gb, act_gb)`：ZeRO-3 下，最小的 2 的幂 N 使 `16P/N/GB + act_gb <= card_gb`。

In [ ]:
def min_gpus(P, card_gb, act_gb=10):
    # TODO: N 从 1 翻倍，直到 16P/N/GB + act_gb <= card_gb
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
P12=param_count(load_config("pythia-12b"))
N=min_gpus(P12, 80)
assert (16*P12/N)/GB + 10 <= 80 and (16*P12/(N//2 if N>1 else 1))/GB + 10 > 80
# 更小的卡需要更多
assert min_gpus(P12, 40) >= min_gpus(P12, 80)
print(f"练习 4 通过 ✓  pythia-12b 训练: 80GB卡需{min_gpus(P12,80)}张, 40GB卡需{min_gpus(P12,40)}张")


---
## 📖 参考答案

In [ ]:
# 练习 1
def zero_per_gpu_gb(P, N, stage):
    m = {1:2*P+2*P+12*P/N, 2:2*P+(2*P+12*P)/N, 3:16*P/N}[stage]
    return m/GB
print("练习 1 ✓")

In [ ]:
# 练习 2
def pipeline_bubble(p, m): return (p-1)/(m+p-1)
def microbatch_for_target(p, target):
    m=1
    while pipeline_bubble(p,m) > target: m+=1
    return m
print("练习 2 ✓")

In [ ]:
# 练习 3
def allreduce_time_ms(P, bw_gbps, dtype_bytes=2):
    return 2*P*dtype_bytes/(bw_gbps*1e9)*1000
print("练习 3 ✓")

In [ ]:
# 练习 4
def min_gpus(P, card_gb, act_gb=10):
    N=1
    while (16*P/N)/GB + act_gb > card_gb: N*=2
    return N
print("练习 4 ✓ —— '多少卡' 是 systems 面试最高频的题")